# Classificação de Materiais Recicláveis utilizando ResNet-50 (Parte 2)
*ESZA019 – Visao Computacional - 2026.2*

Integrantes - Grupo 2:
- Cesar de Jesus Carvalho 
- Mariana Chiara Travassos Sarinho
- Vinícius de Marchi Costa

Este notebook utiliza o modelo ResNet-50 treinado no notebook `01_modelo.ipynb` para realizar a classificação de materiais recicláveis em tempo real. Para isso, duas câmeras serão utilizadas simultaneamente:
- Câmera 1: índice 0,
- Câmera 2: índice 1.

Cada câmera terá seu próprio fluxo de vídeo e as imagens capturadas serão combinadas em uma única imagem de entrada. O sistema apresenta simultaneamente as imagens das duas câmeras e, abaixo delas, mostra a classificação produzida pela ResNet-50.

* Controles:
- `Q`-> encerra o sistema
- `S`-> salva as imagens das duas câmeras e a tela final

* Arquivos necessários:
O Notebook 01 deve ter produzido:
- models/resnet50_waste.keras
- models/class_names.json

------------------
## Configuração do ambiente

In [ ]:
import cv2
import json
import numpy as np
import tensorflow as tf

from pathlib import Path
from datetime import datetime

from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input

A resolução utilizada será de 640 × 480 pixels, seguindo a estrutura
do código de referência.

O parâmetro `PROCESSAR_A_CADA` define a frequência de execução da
ResNet-50.

Por exemplo: 

```text
PROCESSAR_A_CADA = 3

In [ ]:
# ============================================================
# CONFIGURAÇÕES
# ============================================================

CAMERA_1 = 0
CAMERA_2 = 1
LARGURA = 640
ALTURA = 480
PROCESSAR_A_CADA = 3
IMG_SIZE = (224, 224)
LIMIAR_CONFIANCA = 0.50

In [ ]:
# ==========================================================
# PRINCIPAIS CAMINHOS E DIRETÓRIOS
# ==========================================================

# Modelo
MODEL_PATH = Path(
    "models/resnet50_waste.keras"
)

# Classes
CLASS_NAMES_PATH = Path(
    "models/class_names.json"
)


RESULTS_DIR = Path(
    "resultados_webcam"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
if not MODEL_PATH.exists():

    raise FileNotFoundError(
        f"Modelo não encontrado:\n"
        f"{MODEL_PATH}"
    )


if not CLASS_NAMES_PATH.exists():

    raise FileNotFoundError(
        f"Arquivo de classes não encontrado:\n"
        f"{CLASS_NAMES_PATH}"
    )


print("Modelo encontrado:")
print(MODEL_PATH)

print()

print("Arquivo de classes encontrado:")
print(CLASS_NAMES_PATH)

------------------
## Upload da ResNet-50

Nesta etapa será carregado o modelo ResNet-50 treinado no Notebook 01. Não será realizado nenhum novo treinamento. A ResNet-50 será utilizada exclusivamente para inferência, ou seja, para receber uma imagem e produzir uma previsão da classe do material. O modelo será carregado apenas uma vez antes da abertura das câmeras.

In [ ]:
model = load_model(
    MODEL_PATH
)

print(
    "ResNet-50 carregada com sucesso."
)

O modelo retorna um vetor de probabilidades. Por isso, precisamos saber qual classe corresponde a cada posição desse vetor.

O arquivo `class_names.json`, criado no Notebook 01, contém essa relação.

Exemplo:
```text
    0 → cardboard
    1 → glass
    2 → metal
    3 → paper
    4 → plastic
    5 → trash

In [ ]:
with open(CLASS_NAMES_PATH,"r",encoding="utf-8") as arquivo:
    class_names = json.load(arquivo)

print("Classes disponíveis:")

for indice, classe in enumerate(class_names):
    print(f"{indice}: {classe}")

print()
print("Quantidade de classes:",len(class_names))

------------------
## Webcams

A função abaixo segue a estrutura utilizada no código de referência. Cada câmera será aberta utilizando `cv2.VideoCapture()`.

Também serão configurados:
- largura;
- altura;
- tamanho do buffer.

O buffer será mantido pequeno para reduzir o atraso entre a captura da câmera e a imagem exibida na tela.

In [ ]:
def abrir_camera(indice):
    """
    Abre uma câmera utilizando o índice informado.
    Parâmetros
    ----------
    indice : int
        Índice da câmera no sistema.

    Retorno
    -------
    cv2.VideoCapture
        Objeto responsável pela captura dos frames.
    """

    cap = cv2.VideoCapture(indice)

    if cap.isOpened():
        cap.set(cv2.CAP_PROP_FRAME_WIDTH,LARGURA)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT,ALTURA)
        cap.set(cv2.CAP_PROP_BUFFERSIZE,1)

    return cap

### Pré-processamento

A ResNet-50 utilizada no Notebook 01 foi treinada utilizando imagens de tamanho 224 × 224 pixels. Além disso, a função `preprocess_input` da ResNet-50 precisa ser aplicada antes da classificação.

As câmeras, entretanto, fornecem imagens em:
```text
    640 × 480 pixels
    BGR

Portanto, será realizada a seguinte sequência:

```text
    Frame das câmeras
        ↓
    Combinação das duas imagens
        ↓
    BGR → RGB
        ↓
    224 × 224
        ↓
    float32
        ↓
    preprocess_input()
        ↓
    ResNet-50

In [ ]:
def preparar_imagem(imagem):
    """
    Prepara uma imagem para entrada na ResNet-50.
    """

    # Conversão BGR → RGB
    imagem = cv2.cvtColor(imagem,cv2.COLOR_BGR2RGB)

    # Redimensionamento para a entrada da ResNet-50
    imagem = cv2.resize(imagem,IMG_SIZE,interpolation=cv2.INTER_AREA)

    # Conversão para float32
    imagem = imagem.astype(np.float32)

    # Pré-processamento específico da ResNet-50
    imagem = preprocess_input(imagem)

    # Adiciona dimensão do batch
    imagem = np.expand_dims(imagem,axis=0)


    return imagem     

### Combinação das duas webcams

def combinar_cameras(frame1,frame2):
    """
    Combina horizontalmente as imagens
    das duas câmeras.
    """

    # OBTÉM A ALTURA DOS DOIS FRAMES
    altura1 = frame1.shape[0]
    altura2 = frame2.shape[0]

    # DEFINE UMA ALTURA COMUM
    altura = min(altura1,altura2)

    # REDIMENSIONA CAMERA 1
    largura1 = int(frame1.shape[1] * altura / altura1)
    frame1 = cv2.resize(frame1,(largura1,altura),interpolation=cv2.INTER_AREA)

    # REDIMENSIONA CAMERA 2
    largura2 = int(frame2.shape[1] * altura / altura2)

    frame2 = cv2.resize(frame2,(largura2,altura),interpolation=cv2.INTER_AREA)

    # COMBINA HORIZONTALMENTE
    imagem_combinada = np.hstack((frame1,frame2))

    return imagem_combinada

------------------
## Classificação

Nesta etapa será criada a função responsável por realizar a classificação utilizando a ResNet-50. As imagens capturadas pelas duas câmeras serão combinadas em uma única imagem antes da classificação.

Dessa forma, o modelo realizará somente uma previsão:

    Câmera 1 + Câmera 2 → ResNet-50 → uma classe

A função retornará:
- a classe prevista;
- a confiança da previsão.

In [ ]:
def classificar_imagem(imagem):
    """
    Recebe a imagem combinada das duas câmeras,
    realiza uma única classificação com a ResNet-50
    e retorna a classe e a confiança.
    """

    # Pré-processamento
    imagem_processada = preparar_imagem(imagem)

    # Predição
    predicoes = model.predict(
        imagem_processada,
        verbose=0
    )[0]

    # Índice da maior probabilidade
    indice = int(
        np.argmax(predicoes)
    )

    # Confiança
    confianca = float(
        predicoes[indice]
    )

    # Nome da classe
    classe = class_names[indice]

    return classe, confianca

A classificação será apresentada sobre a imagem combinada. Serão mostradas duas informações:
- Classe identificada;
- Confiança da previsão em porcentagem.

A cor do resultado será utilizada apenas para facilitar a visualização:
- verde → confiança igual ou superior ao limite definido;
- laranja → confiança abaixo do limite definido.


def mostrar_classificacao(imagem,classe,confianca):
    """
    Adiciona a classificação diretamente
    sobre a imagem exibida pela webcam.
    """

    porcentagem = confianca * 100

    texto = (f"Classe: {classe} | "f"Confianca: {porcentagem:.1f}%")

    # Cor da classificação
    if confianca >= LIMIAR_CONFIANCA:
        cor = (0, 255, 0)

    else:
        cor = (0, 165, 255)

    # Fundo preto atrás do texto
    (largura_texto, altura_texto), _ = (
        cv2.getTextSize(
            texto,
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            2
        )
    )

    cv2.rectangle(
        imagem,
        (10, 45),
        (
            25 + largura_texto,
            65 + altura_texto
        ),
        (0, 0, 0),
        -1
    )

    # Texto da classificação
    cv2.putText(
        imagem,
        texto,
        (15, 65),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        cor,
        2,
        cv2.LINE_AA
    )

    return imagem

As duas imagens serão apresentadas lado a lado. Para facilitar a identificação de cada fonte de imagem, será inserido um texto indicando:
- CAMERA 1;
- CAMERA 2.

Essa identificação não representa duas classificações diferentes. Ela serve somente para indicar de qual câmera cada imagem foi obtida. A classificação continuará sendo única e será apresentada sobre a imagem combinada.

def identificar_camera(frame,numero):
    """
    Identifica visualmente cada câmera.
    """

    texto = f"CAMERA {numero}"

    cv2.putText(
        frame,
        texto,
        (15, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )

    return frame

### Salvando os resultados

Nesta etapa será criada a função responsável por salvar as imagens capturadas pelas duas câmeras.

Quando a tecla `S` for pressionada, serão salvos:
- frame original da câmera 1;
- frame original da câmera 2;
- imagem combinada das duas câmeras;
- imagem final contendo a classificação.

Os arquivos serão organizados em uma pasta com data e hora dentro de `resultados_webcam/`.

A classificação continua sendo única. Os frames individuais são salvos apenas para registrar as imagens utilizadas pelo sistema.

def salvar_resultados(frame1,frame2,imagem_combinada,tela):
    """
    Salva os frames das duas câmeras,
    a imagem combinada e a tela final.
    """

    # DATA E HORA
    horario = datetime.now().strftime("%Y%m%d_%H%M%S")

    # CRIA PASTA DO RESULTADO
    pasta = (RESULTS_DIR/f"resultado_{horario}")

    pasta.mkdir(parents=True,exist_ok=True)

    # CAMERA 1
    cv2.imwrite(str(pasta / "camera1.jpg"),frame1)

    # CAMERA 2
    cv2.imwrite(str(pasta / "camera2.jpg"),frame2)

    # IMAGEM COMBINADA
    cv2.imwrite(str(pasta / "imagem_combinada.jpg"),imagem_combinada)

    # TELA FINAL
    cv2.imwrite(str(pasta / "tela_completa.jpg"),tela)

    print()
    print("Resultados salvos em:")

    print(pasta.resolve())

### Inicialização das câmeras

In [ ]:
cap1 = abrir_camera(CAMERA_1)
cap2 = abrir_camera(CAMERA_2)

# VERIFICAR CÂMERA 1
if not cap1.isOpened():
    cap1.release()
    cap2.release()
    raise RuntimeError(f"Não foi possível abrir a câmera {CAMERA_1}.")

# VERIFICAR CÂMERA 2
if not cap2.isOpened():
    cap1.release()
    cap2.release()
    raise RuntimeError(f"Não foi possível abrir a câmera {CAMERA_2}.")

print("Câmera 1 aberta com sucesso.")
print("Câmera 2 aberta com sucesso.")

### LOOP PRINCIPAL

In [ ]:
contador = 0
classe_atual = "Aguardando..."
confianca_atual = 0.0

while True:
    # CAPTURA DAS DUAS CÂMERAS
    sucesso1, frame1 = cap1.read()
    sucesso2, frame2 = cap2.read()

    # VERIFICAÇÃO
    if not sucesso1:
        print("Erro ao capturar Camera 1.")
        break

    if not sucesso2:
        print("Erro ao capturar Camera 2.")
        break

    contador += 1

    # GUARDA OS FRAMES ORIGINAIS
    frame1_original = frame1.copy()
    frame2_original = frame2.copy()

    # IDENTIFICAÇÃO DAS CÂMERAS
    frame1_original = frame1.copy()
    frame2_original = frame2.copy()

    # COMBINA AS DUAS IMAGENS
    imagem_combinada = combinar_cameras(frame1_original,frame2_original)

    # CLASSIFICAÇÃO ÚNICA
    if (contador % PROCESSAR_A_CADA == 0 or contador == 1):
        (classe_atual,confianca_atual) = classificar_imagem(imagem_combinada)

    # MOSTRA A CLASSIFICAÇÃO NA IMAGEM
    frame1_exibicao = identificar_camera(frame1_original.copy(),1)
    frame2_exibicao = identificar_camera(frame2_original.copy(),2)

    imagem_combinada = combinar_cameras(frame1_exibicao,frame2_exibicao)
    imagem_combinada = mostrar_classificacao(imagem_combinada,classe_atual,confianca_atual)

    # MOSTRA A IMAGEM AO VIVO
    cv2.imshow("WasteNet - ResNet50 - Duas Cameras",imagem_combinada)

    # TECLADO
    tecla = (cv2.waitKey(1) & 0xFF)

    # Q → SAIR
    if tecla == ord("q"):
        break

    # S → SALVAR
    if tecla == ord("s"):
        salvar_resultados(
            frame1_original,
            frame2_original,
            imagem_combinada,
            imagem_combinada
        )

cap1.release()
cap2.release()
cv2.destroyAllWindows()


print("Câmera 1 liberada.")
print("Câmera 2 liberada.")
print("Sistema encerrado.")